# BME 590 - Workshop 0 - Setup & Introduction
**Professor:** Emma Chory, Ph.D.

**Authors:** 
Rick Wierenga, Joe Laforet, Stefan Golas, Ben Perry

---

---

#### ⚠️ Check your kernel first (top-right corner)

It should read **BME 590 (lab automation)**. If it says *“Select Kernel”*, a Python version, or anything else:

1. Click the kernel picker in the top-right.
2. If VS Code offers **Select Another Kernel…**, click it first. Then choose **Python Environments…** — *not* **Existing Jupyter Server…**, which asks for a URL like `127.0.0.1` and is a dead end.
3. Pick the class environment: the `.venv` inside the class folder. Depending on your VS Code version it is listed as **BME 590 (lab automation)** or as `.venv (Python 3.11)`. (**Jupyter Kernel… → BME 590 (lab automation)** reaches the same environment.)

Wrong kernel = `ModuleNotFoundError` / *"requires the ipykernel package"* on the very first cell. The fix is the 10 seconds above — **do not pip-install anything**.

---


### Usage Note
**Reminder** - You should be running this notebook **locally** on **VS Code** not navigating it through **GitHub**.

---

### Welcome to PyLabRobot!
PyLabRobot *(PLR)* is a universal Python hardware and operating system-agnostic software development kit for automated and autonomous laboratories. PyLabRobot enables control of liquid handling robots, plate readers, pumps, scales, heater shakers, and other equiprment by converting Python commands to their corresponding low-level firmware/IO commands. The primary piece of equipment we care about controlling is the liquid handler, which is a robot that can aspirate and dispense precise volumes of liquid in a Cartesian coordinate system, essentially the same as hand pipetting, but automated!

PLR defines a several universal interface classes:

- **LiquidHandler:** provides generic methods for controlling liquid handlers

- **PlateReader:** (and some other classes) control other equipment like a plate reader. 

These **interface classes** are able to translate singular (**atomic**) commands for a robot (aspirate, drop tips, dispense, move, etc.) to any number of **supported robots** via custom **backends** (or drivers). \[See Figure 1 for a diagram of how this works\] This setup enables the definition of a protocol in Python, subsequent translation to any number of robots, all in one Python script or notebook!

<div>
<img src="../figs/fig_1_plr.jpg" width="1000"/>
</div>

As we work through exercises and tutorials for PLR, make note of these specific help resources! Specifically, reading the **publication below** will help you get oriented with the organization of PLR. For more specific questions, reach out on the class **slack** or to the **TA** or **Professor** via email.

**PLR GitHub:** [Link](https://github.com/PyLabRobot/pylabrobot)

**PLR Forum:** [Link](https://discuss.pylabrobot.org/)

**Publication:** [Link](https://www.cell.com/device/fulltext/S2666-9986\(23\)00170-9)

### Getting Started

---

First, check that PyLabRobot imported cleanly. Running the cell below should print:

```txt
PLR Version:  0.2.2
```

In [ ]:
import pylabrobot

print("PLR Version: ", pylabrobot.__version__)

#### Setting Up Your First Deck & Visualizer

Let's go ahead and set up your first deck for a liquid handler. To set up a liquid handler, you will need three things:

- **Liquid Handler Interface** - This is the top level class that will organize everything else for us.

- **A Deck Layout** - Different machines have different deck sizes/slots. The deck layout tells the LiquidHandler the geometric constraints of our robot.

- **A Backend** -- The backend does the heavy lifting of converting the instructions we write in Python to **machine code** tailored to each robot. Because we do not have a lab component to this class (yet), we must do our experimentation *virtually*. Fortunately, PLR has a built-in `Visualizer` class, which enables us to host a website locally that will display our active experiment as we work!

In the later workshops we will get into the specifics of setting up a deck, and different types of liquid handlers that are available through PLR. For now, we are just trying to make sure you can:

1. Setup a liquid handler deck.

2. Setup the visualizer.

3. Record a GIF of a basic protocol.

Let's build that deck one piece at a time. Each cell below does exactly one thing, and the explanation for it sits right above it. Run them in order -- next week's workshop covers the same ground in more depth.

Start with the imports. `STARLetDeck` is the deck of a Hamilton STARLet; the other four are **resource definitions** -- two carriers, a tip rack, and a plate.

In [ ]:
from pylabrobot.resources import (
    STARLetDeck,
    TIP_CAR_480_A00,
    PLT_CAR_L5AC_A00,
    cor_96_wellplate_360uL_Fb,
    hamilton_96_tiprack_1000uL_filter,
)

Create the deck itself. It is empty apart from the fixtures every STARLet has (the trash, the waste block), and its positions are numbered in **rails** running left to right.

In [ ]:
deck = STARLetDeck()

A **carrier** is a holder that sits on the deck and accepts labware in its slots. This one takes plates. Every resource in PLR needs a `name`, and names must be unique -- that is how you refer to it later.

In [ ]:
plate_carrier = PLT_CAR_L5AC_A00(name="awesome plate carrier")

Creating a resource does not put it anywhere. `assign_child_resource` is what places it, and `rails=5` is where: the carrier starts at rail 5 of the deck.

In [ ]:
deck.assign_child_resource(plate_carrier, rails=5)

The tip carrier is the same two steps -- create it, then place it. It holds tip racks rather than plates, and it goes at rail 11.

In [ ]:
tip_carrier = TIP_CAR_480_A00(name="awesome tip carrier 96x5")

In [ ]:
deck.assign_child_resource(tip_carrier, rails=11)

Now fill the carriers. A carrier's slots are indexed like a list, so `tip_carrier[0] = ...` drops a tip rack into its first slot. This loop fills the first three, each rack getting its own unique name.

In [ ]:
for i in range(3):
    tip_carrier[i] = hamilton_96_tiprack_1000uL_filter(name=f"tip_rack_{i}")

Four plates into the plate carrier, the same way.

In [ ]:
for i in range(4):
    plate_carrier[i] = cor_96_wellplate_360uL_Fb(name=f"plate_{i}")

Nothing above touched a robot or the network -- the whole deck is just objects in memory, which is why none of it needed `await`. The `deck` variable now holds the entire layout. Let's print a summary of it.

In [ ]:
print(deck.summary())

Great! You should get an output that looks like this:

```txt
Rail  Resource                        Type                 Coordinates (mm)
=========================================================================================
(-6)  ├── trash_core96                Trash                (-58.200, 106.000, 216.400)
      │
(5)   ├── awesome plate carrier       PlateCarrier         (190.000, 063.000, 100.000)
      │   ├── plate_0                 Plate                (194.000, 071.500, 183.120)
      │   ├── plate_1                 Plate                (194.000, 167.500, 183.120)
      │   ├── plate_2                 Plate                (194.000, 263.500, 183.120)
      │   ├── plate_3                 Plate                (194.000, 359.500, 183.120)
      │   ├── <empty>
      │
(11)  ├── awesome tip carrier 96x5    TipCarrier           (325.000, 063.000, 100.000)
      │   ├── tip_rack_0              TipRack              (331.200, 073.000, 214.950)
      │   ├── tip_rack_1              TipRack              (331.200, 169.000, 214.950)
      │   ├── tip_rack_2              TipRack              (331.200, 265.000, 214.950)
      │   ├── <empty>
      │   ├── <empty>
      │
(31)  ├── waste_block                 Resource             (775.000, 115.000, 100.000)
      │   ├── teaching_tip_rack       TipRack              (780.900, 461.100, 100.000)
      │   ├── core_grippers           HamiltonCoreGrippers (797.500, 085.500, 205.000)
      │
(32)  ├── trash                       Trash                (800.000, 190.600, 137.100)
```

But this is kind of hard to actually visualize what is going on. To do that, let's setup our `Visualizer`

The **backend** turns PLR commands into instructions for a specific machine. There is no robot on your desk, so we use the chatterbox backend, which simply prints what it would have told the robot to do.

In [ ]:
from pylabrobot.liquid_handling.backends import LiquidHandlerChatterboxBackend

`visualize_deck` is a course helper. It builds the `LiquidHandler` and its `Visualizer` together and attaches the visualizer to the handler as `lh.vis`, so you only have one object to keep track of.

In [ ]:
from bme590.visualizer_ext import visualize_deck

Now start it. Everything above ran in memory, but this genuinely talks to your browser over a websocket -- it is asynchronous, which is why it needs `await`.

In [ ]:
lh = await visualize_deck(deck, LiquidHandlerChatterboxBackend())

Running the above code should output something similar to:

```txt
Websocket server started at http://127.0.0.1:XXXX
File server started at http://127.0.0.1:XXXX . Open this URL in your browser.
```

This is effectively hosting a website at 127.0.0.1, which is the **localhost**, a special IP address that means the server is being hosted on your own machine. This means when you open the URL provided in the output, you are **connecting** to your own machine's server. You should see a chrome (or whatever default browser your machine has) tab open with an output that is similar to this:

<div>
<img src="../figs/carrier_layout.png" width="750"/>
</div>

We aren't giving you the exact photo of the setup, because you will need to submit that as part of your assignment!

If a browser doesn't automatically open, simply **copy/paste** the output URL to your browser of choice. 

The <span style="color:green"><strong>connected</strong></span> text in the top right corner of the visualizer indicates you are actively communicating with your local session. If for some reason this <span style="color:red"><strong>disconnects</strong></span>, the visualizer will need to be reset as it will lose track of any changes.

Congrats! Pylabrobot is working as expected!

---

**TO-DO:** Take a screenshot of your deck setup and save it to submit with the Workshop 0 assignment on Canvas.

---

#### Creating a GIF of a lab protocol

We kinda added everything at once there, but ideally, we would like to create a visualization of any lab protocol we use, **step-by-step**. The classic way to do this with PLR is to click **Start Recording** in the visualizer before running your code, click **Stop Recording** afterwards, then type a filename and click **Download GIF**. That works, but it means tab-switching at exactly the right moments for every single deliverable -- and there are many deliverables in this course.

Instead, every workshop uses a small course helper that records **from your code**:

```python
rec = gif_recorder(lh.vis, name="my_protocol.gif")

await rec.start()
# ... deck changes and pipetting ...
await rec.stop()   # GIF renders and downloads automatically
```

Everything the deck does between `rec.start()` and `rec.stop()` is captured frame-by-frame, and the finished GIF **downloads automatically** under the exact filename you gave. No buttons, no tab switching, no renaming.

**Where the file goes:** your *browser* saves it, not Python, so it lands in whatever folder your browser downloads to -- usually `Downloads`, wherever that is on your machine. The notebook cannot put it anywhere else, and `name` is a **filename, not a path**: browsers flatten path separators, so `runs/out.gif` saves as `runs_out.gif` and `C:\out\run.gif` as `C__out_run.gif`. Pass a plain name like `lab_0_deck_setup.gif` and move the file afterwards if you want it somewhere else. (`.gif` is added for you if you leave it off.)

A few other things worth knowing:

1. `rec.start()` waits for the visualizer page to connect before recording begins, so you don't need artificial delays to go click a button first.

2. You still control pacing yourself, with `await step()` from the same module. A GIF is built from what the deck looked like at each moment, so without a pause between changes they all land on one frame. `step()` inserts that pause; `step(2)` waits twice as long. Every workshop from here on paces its protocols this way.

3. If you forget `rec.stop()`, the recorder stops itself after **60 seconds** and downloads what it captured -- so a forgotten stop costs a short GIF, not a lost one.

4. The **Start/Stop Recording** toolbar buttons still work exactly as before if you ever want a manual, ad-hoc recording -- the two approaches don't interfere.

5. Generally, it is good practice to work first without the visualizer as much as possible. The reason is because if an error occurs while you are using the visualizer, you may have to **reset** both the **deck** and **visualizer**. Every time the visualizer is reset, it will open a **new tab**, which can become annoying after several iterations.

    - It is also good to **functionalize your code** to have one function which **sets up your deck** and one which **runs the protocol**. Typically you will find most of your errors will occur in coding the protocol, so having a function to easily set up your deck again will **speed up your dvelopment**

    - We will cover this more in the deck setup tutorial.



You just placed every piece of that deck by hand, one cell at a time, which is the way to learn what each call does -- but not the way to run a protocol you intend to repeat. So this time the same deck gets built by a single function, recorded from start to finish.

Start over with an empty deck. The visualizer is set up *before* anything is on it, so you can watch it fill up.

**One visualizer at a time.** The cell below prints a note saying it closed the previous visualizer, and opens a **new browser tab** -- the old tab goes dead, so close it. This matters more than it looks: the visualizer's web page is served on a fixed port, so two live sessions mean the tab you are looking at belongs to the *first* one while your code drives the second, and a recording started on it would wait for a browser that never arrives. If you ever want to shut the visualizer down without starting a new deck, run `await close_visualizer()` (import it from `bme590.visualizer_ext`).

In [ ]:
deck = STARLetDeck()
lh = await visualize_deck(deck, LiquidHandlerChatterboxBackend())

Now start recording. Everything that happens to the deck from here until `rec.stop()` ends up in `lab_0_deck_setup.gif`.

In [ ]:
from bme590.visualizer_ext import gif_recorder

rec = gif_recorder(lh.vis, name="lab_0_deck_setup.gif")
await rec.start()

The recorder is running, so now build the deck. This is where the **functionalize your deck setup** advice pays off: instead of a cell per change, the whole layout lives in one function you can call again any time you need a fresh deck.

The pacing comes from `await step()`, which pauses long enough for the current state to land on its own GIF frame. Without it the deck would appear fully built in a single frame. `step(2)` waits twice as long, `step(0.5)` half.

In [ ]:
from bme590.visualizer_ext import step


async def build_deck(deck):
    plate_carrier = PLT_CAR_L5AC_A00(name="awesome plate carrier")
    deck.assign_child_resource(plate_carrier, rails=5)
    await step()

    tip_carrier = TIP_CAR_480_A00(name="awesome tip carrier 96x5")
    deck.assign_child_resource(tip_carrier, rails=11)
    await step()

    for i in range(3):
        tip_carrier[i] = hamilton_96_tiprack_1000uL_filter(name=f"tip_rack_{i}")
        await step()

    for i in range(4):
        plate_carrier[i] = cor_96_wellplate_360uL_Fb(name=f"plate_{i}")
        await step()

Defining the function changed nothing on the deck. Run it, then watch the visualizer tab: the carriers appear, then the tip racks and plates drop in one at a time.

In [ ]:
await build_deck(deck)

The deck is complete. Stop the recording -- the GIF renders and downloads automatically.

In [ ]:
await rec.stop()
print("Done! lab_0_deck_setup.gif should appear in your Downloads folder.")

If done correctly, the deck was built piece by piece while being recorded, and `lab_0_deck_setup.gif` downloaded automatically to your browser's **Downloads folder** when `rec.stop()` ran. Open it and verify it shows the carriers, racks, and plates appearing step by step.

---

**TO-DO:** Submit `lab_0_deck_setup.gif` with the Workshop 0 assignment on Canvas. If the download did not start, make sure the visualizer tab was open and showing **connected** in the top right corner, then run the cell again.

---


#### Conclusion

That's all for workshop 0! As long as you were able to successfully download both a screenshot of your final deck setup and GIF, you should be good to go. 

For the remainder of the workshops, we will cover more technical aspects of PLR, but in general, all of the assignments will require you to submit one or more of the following:

1. A `.gif` file of your protocol running.

2. A `.png` or `.jpeg` file of your initial deck setup or final deck setup.

3. A write-up describing your protocol, and any extensions thereof.

4. A `.ipynb`, `.py`, or `.txt` file with your final code to produce that protocol.

For this assignment, you just need to submit the `.png` file and the `.gif` file. If you are still feeling unsure on how to generate any of the following, please **reach out to the teaching team**, contact info for whom can be found in the .`README.md` file on the [class GitHub](https://github.com/chory-lab/bme590-fall-2026)

---